# 01 - Exploratory Data Analysis (EDA)

**Role: Data Analyst**

**Goal:** Understand the crime data structure and build intuition before modeling.

---

## Objectives

1. Load and inspect NYPD complaint data
2. Understand temporal range and crime categories
3. Identify violent crime subset
4. Check data quality (missing values, outliers)
5. Visualize temporal and spatial patterns

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set styles
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

%matplotlib inline

## 1. Load Data

In [ ]:
# Load NYPD crime data
data_path = Path('../NYPD_Complaint_Data_Historic.csv')

# Read first few rows to check structure
df_sample = pd.read_csv(data_path, nrows=1000)
print(f"Shape (sample): {df_sample.shape}")
df_sample.head()

In [ ]:
# Check column names and types
df_sample.info()

In [ ]:
# Identify key columns for our analysis
# We need: date, crime type, coordinates (lat/lon)
print("Columns available:")
print(df_sample.columns.tolist())

## 2. Data Quality Assessment

In [ ]:
# Check for missing values in key columns
key_cols = ['CMPLNT_FR_DT', 'OFNS_DESC', 'Latitude', 'Longitude']
missing = df_sample[key_cols].isnull().sum()
print("Missing values:")
print(missing)
print(f"\nMissing percentage:")
print(missing / len(df_sample) * 100)

## 3. Temporal Range

In [ ]:
# Parse dates and check temporal coverage
df_sample['date'] = pd.to_datetime(df_sample['CMPLNT_FR_DT'], errors='coerce')
df_sample['year'] = df_sample['date'].dt.year

print(f"Date range: {df_sample['date'].min()} to {df_sample['date'].max()}")
print(f"\nYears covered:")
print(df_sample['year'].value_counts().sort_index())

## 4. Crime Categories

**Focus: Violent Crime** (homicide, assault, robbery, battery)

In [ ]:
# Check crime type descriptions
print("Top 20 crime categories:")
print(df_sample['OFNS_DESC'].value_counts().head(20))

In [ ]:
# Define violent crime categories
violent_crimes = [
    'MURDER & NON-NEGL. MANSLAUGHTER',
    'RAPE',
    'ROBBERY',
    'FELONY ASSAULT',
    'ASSAULT 3 & RELATED OFFENSES'
]

# Filter violent crimes
df_violent = df_sample[df_sample['OFNS_DESC'].isin(violent_crimes)]
print(f"\nViolent crimes: {len(df_violent)} ({len(df_violent)/len(df_sample)*100:.1f}%)")
print("\nViolent crime breakdown:")
print(df_violent['OFNS_DESC'].value_counts())

## 5. Spatial Coverage

In [ ]:
# Check coordinate coverage
coords_available = df_sample[['Latitude', 'Longitude']].notna().all(axis=1)
print(f"Records with coordinates: {coords_available.sum()} ({coords_available.sum()/len(df_sample)*100:.1f}%)")

# Basic spatial bounds
df_with_coords = df_sample[coords_available]
print(f"\nLatitude range: {df_with_coords['Latitude'].min():.4f} to {df_with_coords['Latitude'].max():.4f}")
print(f"Longitude range: {df_with_coords['Longitude'].min():.4f} to {df_with_coords['Longitude'].max():.4f}")

In [ ]:
# Quick scatter plot to verify coordinates
plt.figure(figsize=(10, 10))
plt.scatter(
    df_with_coords['Longitude'], 
    df_with_coords['Latitude'],
    alpha=0.1,
    s=1
)
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Crime Locations (Sample)')
plt.show()

## Next Steps

1. Load **full dataset** (filtered for 2018-2022)
2. Obtain **NYC census tract shapefiles**
3. Download **ACS socio-economic data**
4. Build **spatial join pipeline**